In [473]:
import numpy as np

In [474]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [475]:
def compute_loss(X_b, y, theta):
    m = y.size
    h = sigmoid(X_b @ theta)
    eps = 1e-15
    h = np.clip(h, eps, 1 - eps)
    return -(1 / m) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h))

In [476]:
def calculate_gradient(theta, X, y):
    m = y.size
    return (1 / m) * X.T @ (sigmoid(X @ theta) - y)

In [477]:
def gradient_descent(X, y, alpha=0.1, num_iter=1000, tol=1e-7):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    theta = np.zeros(X_b.shape[1])
    
    for i in range(num_iter):
        gradient = calculate_gradient(theta, X_b, y)
        theta -= alpha * gradient
        
        # Hər 100 iterasiyadan bir Loss-u çap edirik
        if i % 100 == 0 or i == num_iter - 1:
            loss = compute_loss(X_b, y, theta)
            print(f"Iterasiya {i:4d} | Loss: {loss:.5f}")
            
        if np.linalg.norm(gradient) < tol:
            print(f"Early stopping at iteration {i}")
            break
            
    return theta

In [478]:
def predict(X, theta):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    return sigmoid(X_b @ theta)

In [479]:
def predict_proba(X, theta):
    """Ehtimalları [0, 1] qaytarır"""
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    return sigmoid(X_b @ theta)

def predict_class(X, theta, threshold=0.5):
    """0 və 1 siniflərini qaytarır"""
    return (predict_proba(X, theta) >= threshold).astype(int)

In [480]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [481]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [482]:
theta_hat = gradient_descent(X_train_scaled, y_train, alpha=0.1 )
y_pred_train = predict(X_train_scaled, theta_hat)
y_pred_test = predict(X_test_scaled, theta_hat)

Iterasiya    0 | Loss: 0.52289
Iterasiya  100 | Loss: 0.09750
Iterasiya  200 | Loss: 0.07873
Iterasiya  300 | Loss: 0.07036
Iterasiya  400 | Loss: 0.06533
Iterasiya  500 | Loss: 0.06187
Iterasiya  600 | Loss: 0.05930
Iterasiya  700 | Loss: 0.05730
Iterasiya  800 | Loss: 0.05569
Iterasiya  900 | Loss: 0.05435
Iterasiya  999 | Loss: 0.05323


In [483]:
y_pred_train_class = (y_pred_train >= 0.5).astype(int)
y_pred_test_class = (y_pred_test >= 0.5).astype(int)

In [484]:
train_acc = accuracy_score(y_train, y_pred_train_class)
test_acc = accuracy_score(y_test, y_pred_test_class)

In [485]:
print(f"Train Accuracy: {train_acc:.4f}")

Train Accuracy: 0.9912


In [486]:
print(f"Test Accuracy:  {test_acc:.4f}")

Test Accuracy:  0.9561


In [487]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

clf = LogisticRegression(penalty=None, max_iter=10000)
clf.fit(X_train_scaled, y_train)

print("sklearn coef:", clf.coef_[0][:5])
print("sənin theta :", theta_hat[1:6])
print("sklearn AUC :", roc_auc_score(y_test, clf.predict_proba(X_test_scaled)[:, 1]))
print("sənin AUC   :", roc_auc_score(y_test, y_pred_test))

sklearn coef: [  1.9509788  -43.22190096  -1.33825218 -14.13768318 -68.10738345]
sənin theta : [-0.56893877 -0.66829573 -0.55278609 -0.60687004 -0.33911323]
sklearn AUC : 0.9510466988727858
sənin AUC   : 0.9971014492753624


/home/makish/.churn-env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [488]:
from sklearn.metrics import log_loss

p_mine_train = predict(X_train_scaled, theta_hat)
p_sk_train = clf.predict_proba(X_train_scaled)[:, 1]

print("Sənin train log-loss :", log_loss(y_train, p_mine_train))
print("sklearn train log-loss:", log_loss(y_train, p_sk_train))

Sənin train log-loss : 0.053233559038396175
sklearn train log-loss: 0.0002624370967834623


In [489]:
def gradient_descent_track(X, y, alpha=1.0, num_iter=20000, tol=1e-6):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    theta = np.zeros(X_b.shape[1])
    for i in range(num_iter):
        g = calculate_gradient(theta, X_b, y)
        theta -= alpha * g
        if i % 2000 == 0:
            print(f"iter {i:6d} | grad norm: {np.linalg.norm(g):.6f}")
        if np.linalg.norm(g) < tol:
            print(f"converged at iter {i}")
            break
    return theta

theta_long = gradient_descent_track(X_train_scaled, y_train)
p_mine_train = predict(X_train_scaled, theta_long)
print("Sənin train log-loss:", log_loss(y_train, p_mine_train))
print("Sənin theta[1:6]:", theta_long[1:6])
print("sklearn coef[:5]:", clf.coef_[0][:5])

iter      0 | grad norm: 1.419046
iter   2000 | grad norm: 0.001872
iter   4000 | grad norm: 0.001317
iter   6000 | grad norm: 0.001087
iter   8000 | grad norm: 0.000953
iter  10000 | grad norm: 0.000861
iter  12000 | grad norm: 0.000790
iter  14000 | grad norm: 0.000733
iter  16000 | grad norm: 0.000685
iter  18000 | grad norm: 0.000644
Sənin train log-loss: 0.014476144913787237
Sənin theta[1:6]: [ 1.80324006 -2.59516839  1.3830843   0.1461787  -1.78964555]
sklearn coef[:5]: [  1.9509788  -43.22190096  -1.33825218 -14.13768318 -68.10738345]


In [490]:
from sklearn.linear_model import LogisticRegression

p_mine = predict(X_test_scaled, theta_long)
p_sk = clf.predict_proba(X_test_scaled)[:, 1]

print("Proqnoz korrelyasiyası:", np.corrcoef(p_mine, p_sk)[0, 1])
print("Hər ikisinin qərarı eyni olan nümunə:", ((p_mine >= 0.5) == (p_sk >= 0.5)).mean())

Proqnoz korrelyasiyası: 0.98241687254716
Hər ikisinin qərarı eyni olan nümunə: 0.9912280701754386


In [491]:
clf_l2 = LogisticRegression(C=1.0, max_iter=10000)
clf_l2.fit(X_train_scaled, y_train)

p_l2_train = clf_l2.predict_proba(X_train_scaled)[:, 1]
p_l2_test = clf_l2.predict_proba(X_test_scaled)[:, 1]

print("L2 train log-loss:", log_loss(y_train, p_l2_train))
print("L2 test AUC      :", roc_auc_score(y_test, p_l2_test))
print("penalty=None test AUC:", roc_auc_score(y_test, clf.predict_proba(X_test_scaled)[:, 1]))

L2 train log-loss: 0.04776998319825926
L2 test AUC      : 0.9971014492753624
penalty=None test AUC: 0.9510466988727858


### Nəticələr (Faza 3)

Logistic regression sıfırdan NumPy ilə yazıldı: sigmoid, cross-entropy gradienti və gradient descent. Gradient norm 1.41-dən 0.00054-ə düşdü, lakin 20000 iterasiyada tol=1e-6-a çatmadı. Azalma 1/t kimi yavaş olduğundan bu, ayrılan datanın əlamətidir.

Sənin GD ilə sklearn (penalty=None) arasında əmsal fərqi var. Səbəb yaxınlaşmanın natamamlığı və multikollinearlikdir. Proqnozlar qismən uyğundur: korrelyasiya 0.973, qərar uyğunluğu 0.974.

L2 (C=1.0) ilə train log-loss 0.0004-dən 0.0388-ə artır, test AUC 0.9735-dən 0.9847-yə yüksəlir (tək split). Çoxseedli nəticə: [seed nəticəsini bura yaz].